# MovieLens Bootcamp — Demo Notebook

Этот ноутбук — **сценарий-презентация** для защиты проекта (Movies / Ratings / Tags / Links).
Он показывает, что код запускается, и демонстрирует ключевые методы на первых 1000 строках датасетов.

> Подстрой пути в секции **Paths** под свою структуру проекта.

## План демо (3–5 минут)

1) Инициализация классов и загрузка данных  
2) **Movies:** годы релиза, жанры, фильмы с максимальным числом жанров  
3) **Ratings:** распределения, топы по количеству/средней/спорности  
4) **Users:** активность пользователей и “спорность”  
5) **Tags:** популярность и поиск по слову  
6) **Links + IMDb:** режиссёры, бюджеты, прибыль, длительность, cost/min  
7) Выводы и идеи улучшений

## 0) Setup

In [ ]:
from collections import Counter, defaultdict
import os
import requests
from bs4 import BeautifulSoup

# Если твои классы лежат в отдельном файле, импортируй отсюда:
# from your_module import Movies, Ratings, Tags, Links

# Если ты вставляешь классы прямо в ноутбук — просто пропусти импорт выше.

## Paths

In [ ]:
# Вариант 1: датасеты лежат в папке ../datasets (как часто в заданиях)
DATA_DIR = "../datasets"

# Вариант 2: датасеты лежат рядом с ноутбуком
# DATA_DIR = "."

PATH_MOVIES  = os.path.join(DATA_DIR, "movies.csv")
PATH_RATINGS = os.path.join(DATA_DIR, "ratings.csv")
PATH_TAGS    = os.path.join(DATA_DIR, "tags.csv")
PATH_LINKS   = os.path.join(DATA_DIR, "links.csv")

for p in [PATH_MOVIES, PATH_RATINGS, PATH_TAGS, PATH_LINKS]:
    print(p, "✅" if os.path.exists(p) else "❌ NOT FOUND")

## Инициализация объектов

In [ ]:
movies = Movies(PATH_MOVIES)
ratings = Ratings(PATH_RATINGS)

# Внутренние классы
ratings_movies = ratings.Movies(ratings)
ratings_users  = ratings.Users(ratings)

tags = Tags(PATH_TAGS)
links = Links(PATH_LINKS)

print("Movies rows:", len(movies.arr_of_movies))
print("Ratings rows:", len(ratings.arr_of_ratings))
print("Tags rows:", len(tags.arr_of_tags))
print("Links rows:", len(links.arr_of_links))
print("IMDb rows:", len(links.data_imdb))

--- 
# 1) Movies — анализ `movies.csv`
Файл содержит `movieId,title,genres`.

Покажем:
- распределение по годам релиза (год из названия)
- распределение по жанрам
- топ фильмов по количеству жанров

## 1.1 Распределение по годам релиза

In [ ]:
release_dist = movies.dist_by_release()
list(release_dist.items())[:10]

## 1.2 Распределение по жанрам

In [ ]:
genres_dist = movies.dist_by_genres()
list(genres_dist.items())[:10]

## 1.3 Топ по количеству жанров

In [ ]:
movies.most_genres(10)

---
# 2) Ratings — анализ `ratings.csv`
Файл содержит `userId,movieId,rating,timestamp`.

Покажем:
- как менялась активность по годам
- какие оценки встречаются чаще
- топ фильмов по количеству оценок
- топ фильмов по средней оценке
- “самые спорные” фильмы (variance)

## 2.1 Распределение оценок по годам

In [ ]:
year_dist = ratings_movies.dist_by_year()
# первые и последние годы (чтобы увидеть диапазон)
list(year_dist.items())[:10], list(year_dist.items())[-10:]

## 2.2 Распределение по значениям рейтингов

In [ ]:
rating_dist = ratings_movies.dist_by_rating()
list(rating_dist.items())[:15]

## 2.3 Топ фильмов по количеству оценок

In [ ]:
ratings_movies.top_by_num_of_ratings(10)

## 2.4 Топ фильмов по средней оценке (average)

In [ ]:
# metric по умолчанию average
ratings_movies.top_by_ratings(10)

## 2.5 Топ спорных фильмов (variance)

In [ ]:
ratings_movies.top_controversial(10)

---
# 3) Users — анализ поведения пользователей
Покажем:
- кто оставляет больше всего оценок
- у кого выше средняя оценка
- кто ставит максимально противоречивые оценки (variance)

## 3.1 Топ пользователей по количеству оценок

In [ ]:
ratings_users.users_top_by_num_of_ratings(10)

## 3.2 Топ пользователей по средней оценке

In [ ]:
ratings_users.users_top_by_ratings(10)

## 3.3 Топ “спорных” пользователей

In [1]:
ratings_users.users_top_controversial(10)

NameError: name 'ratings_users' is not defined

---
# 4) Tags — анализ `tags.csv`
Файл содержит пользовательские теги к фильмам.

Покажем:
- теги с максимальным числом слов
- самые длинные теги по символам
- пересечение (длинные и многословные)
- популярные теги
- поиск тегов по слову

## 4.1 Теги с максимальным числом слов

In [ ]:
tags.most_words(10)

## 4.2 Самые длинные теги по символам

In [ ]:
tags.longest(10)

## 4.3 Пересечение: многословные ∩ длинные

In [ ]:
tags.most_words_and_longest(10)

## 4.4 Самые популярные теги

In [ ]:
tags.most_popular(10)

## 4.5 Поиск тегов по слову

In [ ]:
tags.tags_with("funny")[:25]

---
# 5) Links + IMDb — доп. бизнес-метрики
`links.csv` связывает movieId ↔ imdbId.

У тебя также есть файл `data_imdb.csv` (или он создаётся парсером IMDb).
Покажем:
- топ режиссёров по числу фильмов
- самые дорогие фильмы (budget)
- самые прибыльные (gross - budget)
- самые длинные (runtime)
- стоимость минуты (budget / runtime)

## 5.1 Топ режиссёров

In [ ]:
links.top_directors(10)

## 5.2 Самые дорогие и самые прибыльные

In [ ]:
links.most_expensive(10), links.most_profitable(10)

## 5.3 Самые длинные и cost/min

In [ ]:
links.longest(10), links.top_cost_per_minute(10)

---
# 6) Выводы (что сказать на защите)

- Код загружает данные и даёт базовую статистику по каталогу фильмов  
- Ratings позволяют увидеть активность по годам и распределение оценок  
- Можно находить топы (по количеству, по средней, по спорности)  
- Tags показывают “язык” аудитории (что люди пишут чаще)  
- IMDb-данные добавляют бизнес-смысл: бюджеты, сборы, прибыльность, длительность, cost/min

## Идеи улучшений (если спросят)
- аккуратнее парсить год из title (не всегда есть `(YYYY)` в конце)  
- год из timestamp лучше через `datetime`  
- связать `movieId → title`, чтобы в топах печатать названия вместо id  
- для IMDb-скрапера: кэш, задержки, обработка 429, таймауты